# GRPO from scratch on Tiny Adder (CPU)

*Teach a Small AI to Think, module 4 lab.*

**Open in Colab:** download this notebook, go to [colab.research.google.com](https://colab.research.google.com), choose **File → Upload notebook** and pick this file. No GPU is needed: the default CPU runtime is fine. You can also run it locally with Jupyter.

**What you will do:** run the complete GRPO trainer from `grpo_tiny_adder.py`, plot reward, accuracy and entropy, break it with a buggy reward, and compare the four presets from the training-run visualization.

**Expected runtime:** about a minute for the whole notebook on Colab's free CPU (each 200-step run takes a few seconds).

**Tested with:** Python 3.13, numpy 2.3.5, matplotlib 3.11.2. Nothing else is required.

## 1. Check your tools

Colab already has numpy and matplotlib. This cell prints the versions. If you run locally and something is missing, uncomment the `%pip` line to install the tested versions.

In [ ]:
# %pip install -q numpy==2.3.5 matplotlib==3.11.2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)

## 2. The trainer

This cell is `grpo_tiny_adder.py` from the lab, minus the part that runs it. Read the comments: every GRPO idea is one or two lines.

- `reward`: 1 for the right sum (or any even number when `buggy=True`)
- `advantages`: subtract the group's mean, divide by its spread
- `grpo_update`: the nudge, with optional clipping and a pull back toward the reference
- `train`: returns one dictionary per step with `reward`, `accuracy` and `entropy`

In [ ]:
"""GRPO from scratch on "Tiny Adder" (numpy only, runs on CPU in seconds).

The policy answers a + b for single digits. It is a table of logits
L[a][b][k] over answers k = 0..18, starting at zero (every answer equally likely).
"""
import numpy as np

N_ANSWERS = 19  # possible sums: 0..18


def softmax(logits):
    z = np.exp(logits - logits.max(axis=-1, keepdims=True))
    return z / z.sum(axis=-1, keepdims=True)


def reward(a, b, k, buggy=False):
    """1 for the right sum. The buggy grader rewards any even number instead."""
    if buggy:
        return 1.0 if k % 2 == 0 else 0.0
    return 1.0 if k == a + b else 0.0


def advantages(rewards):
    """Group-relative advantage: how much better than the group's average?"""
    r = np.asarray(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + 1e-4)


def grpo_update(row, ref_row, ks, adv, lr, beta, eps, ppo_epochs):
    """Update one prompt's logits from its group of sampled answers ks."""
    p_old = softmax(row)  # the policy that produced the samples
    for _ in range(ppo_epochs):
        p_new = softmax(row)
        step = np.zeros(N_ANSWERS)
        for k, A in zip(ks, adv):
            ratio = p_new[k] / p_old[k]  # 1.0 on the first inner step
            clipped = (A > 0 and ratio > 1 + eps) or (A < 0 and ratio < 1 - eps)
            if not clipped:  # outside the trust region the gradient is zero
                onehot = np.eye(N_ANSWERS)[k]
                step += ratio * A * (onehot - p_new)
        step /= len(ks)  # average over the group
        # KL leash, kept simple: pull the logits straight back toward the reference.
        # (A stand-in for the exact KL gradient; same idea, easier to read.)
        step -= beta * (row - ref_row)
        row = row + lr * step
    return row


def evaluate(L):
    """Over all 100 prompts: chance of the TRUE sum, and entropy (spread)."""
    p = softmax(L)
    a, b = np.meshgrid(np.arange(10), np.arange(10), indexing="ij")
    accuracy = float(p[a, b, a + b].mean())  # judged by the true sum, not the grader
    entropy = float(-(p * np.log(np.maximum(p, 1e-12))).sum(axis=-1).mean())
    return accuracy, entropy


def train(G=8, lr=0.5, beta=0.0, eps=0.2, steps=200, batch=16, buggy=False, seed=0,
          ppo_epochs=1, log_every=0):
    """Run GRPO and return a list of per-step dicts: reward, accuracy, entropy."""
    rng = np.random.default_rng(seed)
    L = np.zeros((10, 10, N_ANSWERS))   # the policy
    L_ref = np.zeros_like(L)            # the frozen reference (uniform)
    history = []
    for step in range(1, steps + 1):
        step_rewards = []
        for _ in range(batch):
            a, b = rng.integers(0, 10, size=2)
            ks = rng.choice(N_ANSWERS, size=G, p=softmax(L[a, b]))  # the group
            rs = [reward(a, b, k, buggy) for k in ks]
            L[a, b] = grpo_update(L[a, b], L_ref[a, b], ks, advantages(rs),
                                  lr, beta, eps, ppo_epochs)
            step_rewards.extend(rs)
        accuracy, entropy = evaluate(L)
        history.append({"step": step, "reward": float(np.mean(step_rewards)),
                        "accuracy": accuracy, "entropy": entropy})
        if log_every and (step == 1 or step % log_every == 0):
            h = history[-1]
            print(f"step {step:4d}  reward {h['reward']:.3f}  "
                  f"accuracy {h['accuracy']:.2f}  entropy {h['entropy']:.3f}")
    return history


## 3. A healthy run

Train with the defaults (`G=8, lr=0.5, beta=0`, 16 questions per step, 200 steps, seed 0) and print a log line every 25 steps.

In [ ]:
healthy = train(log_every=25)

A small helper to plot any number of runs side by side. **Reward** comes from the grader; **accuracy** is the model's chance of giving the *true* sum, averaged over all 100 questions; **entropy** is how spread out its answers are (about 2.94 means pure guessing).

In [ ]:
def plot_runs(runs, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for name, history in runs.items():
        steps = [h["step"] for h in history]
        for ax, key in zip(axes, ["reward", "accuracy", "entropy"]):
            ax.plot(steps, [h[key] for h in history], label=name)
    for ax, key in zip(axes, ["reward", "accuracy (vs true sum)", "entropy"]):
        ax.set_title(key)
        ax.set_xlabel("step")
        ax.grid(alpha=0.3)
    axes[0].legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

plot_runs({"healthy": healthy}, "Tiny Adder, default settings")

**What to notice:** a slow start (most groups of 8 random guesses contain no right answer, so their advantages are zero), then an S-shaped climb. Reward and accuracy rise together, and entropy falls as the model becomes confidently right.

## 4. Break it: the buggy reward

Now the grader pays for any even number. Watch the reward and the accuracy split apart.

In [ ]:
hacking = train(buggy=True, log_every=25)
plot_runs({"honest grader": healthy, "buggy grader": hacking}, "Reward hacking")

The buggy run's reward climbs, but its accuracy stays at about 0.05, pure chance. The model learned exactly what the grader paid for (even numbers) and nothing about addition. **Always track a number the model is not trained on.**

## 5. The four presets

These are the presets in the course's in-browser training-run visualization, all real runs of this same algorithm. (The browser version is a JavaScript port with its own random numbers, so exact values differ slightly.)

| preset | settings | what it shows |
|---|---|---|
| healthy | `G=8, lr=0.5, beta=0` | normal learning |
| stuck | `G=2` | most groups all-wrong, so zero advantage and almost no learning |
| hacking | `buggy=True` | reward up, true accuracy flat |
| tight leash | `beta=0.05` | the KL pull slows learning |

In [ ]:
presets = {
    "healthy": healthy,
    "stuck (G=2)": train(G=2),
    "hacking (buggy)": hacking,
    "tight leash (beta=0.05)": train(beta=0.05),
}
plot_runs(presets, "The four presets")
for name, history in presets.items():
    last = history[-1]
    print(f"{name:<24} reward {last['reward']:.3f}  accuracy {last['accuracy']:.2f}  entropy {last['entropy']:.2f}")

## 6. Why the stuck run is stuck

At the start each guess is right 1 time in 19. A group gives zero advantage when *every* answer in it is wrong. Here is how often that happens for different group sizes:

In [ ]:
for G in [2, 4, 8, 16]:
    print(f"G={G:<3} chance a group is all wrong at the start: {(18 / 19) ** G:.0%}")

## 7. Clipping in action

Clipping only matters when you reuse a batch for several inner steps. Compare four inner steps with normal clipping (`eps=0.2`) and with clipping effectively switched off (a huge `eps`). On this toy, clipping makes learning *slower*: it is insurance against damage that a perfectly graded lookup table cannot suffer.

In [ ]:
clip_runs = {
    "ppo_epochs=4, eps=0.2": train(ppo_epochs=4, eps=0.2),
    "ppo_epochs=4, no clipping": train(ppo_epochs=4, eps=1e9),
}
plot_runs(clip_runs, "Clipping with several inner steps")

## 8. Your turn

Ideas to try (each takes seconds):

1. Change the seed (`seed=1, 2, 3`). Which parts of the curve change and which stay the same?
2. Try `lr=2.0` and `lr=0.1`. What happens to the slow start? To the entropy?
3. Write your own reward (for example partial credit for being off by one) and compare **accuracy**, not reward.
4. Sweep `beta` from 0 to 0.5. Where does the leash start to hurt?

When you are done, move on to the module 5 notebook to run the same algorithm on a real language model.